In [1]:
!wget https://huggingface.co/datasets/barryallen16/Sarav-real-fake-automobile-parts-dataset/resolve/main/dataset1.zip

--2026-02-05 10:13:54--  https://huggingface.co/datasets/barryallen16/Sarav-real-fake-automobile-parts-dataset/resolve/main/dataset1.zip
Resolving huggingface.co (huggingface.co)... 3.170.185.35, 3.170.185.14, 3.170.185.25, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.35|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/68dc24c5ee15b4bd54002f0b/ec5afaf1bdc695b853ddf32deffe40de6cbe55f48ccb0a2bc7c07690576f5fb4?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dataset1.zip%3B+filename%3D%22dataset1.zip%22%3B&response-content-type=application%2Fzip&Expires=1770290034&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzcwMjkwMDM0fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjhkYzI0YzVlZTE1YjRiZDU0MDAyZjBiL2VjNWFmYWYxYmRjNjk1Yjg1M2RkZjMyZGVmZmU0MGRlNmNiZTU1ZjQ4Y2NiMGEyYmM3YzA3NjkwNTc2ZjVmYjRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVz

In [2]:
!unzip dataset1.zip -d unzipped/

Archive:  dataset1.zip
   creating: unzipped/dataset1/
   creating: unzipped/dataset1/air filter fake/
  inflating: unzipped/dataset1/air filter fake/10.jpg  
  inflating: unzipped/dataset1/air filter fake/11.jpg  
  inflating: unzipped/dataset1/air filter fake/12.jpg  
  inflating: unzipped/dataset1/air filter fake/13.jpg  
  inflating: unzipped/dataset1/air filter fake/14.jpg  
  inflating: unzipped/dataset1/air filter fake/15.jpg  
  inflating: unzipped/dataset1/air filter fake/17.jpg  
  inflating: unzipped/dataset1/air filter fake/18.jpg  
  inflating: unzipped/dataset1/air filter fake/19.jpg  
  inflating: unzipped/dataset1/air filter fake/2.jpg  
  inflating: unzipped/dataset1/air filter fake/20.jpg  
  inflating: unzipped/dataset1/air filter fake/21.jpg  
  inflating: unzipped/dataset1/air filter fake/22.jpg  
  inflating: unzipped/dataset1/air filter fake/23.jpg  
  inflating: unzipped/dataset1/air filter fake/24.jpg  
  inflating: unzipped/dataset1/air filter fake/25.jpg  
  

In [34]:
import os
os.listdir("unzipped/")

['all_helmet_images', 'all_airfilter_images', 'all_sparkplug_images']

In [26]:
!mkdir unzipped/all_sparkplug_images/

In [29]:
!mv "unzipped/dataset1/spark plug fake"/* unzipped/all_sparkplug_images/

In [33]:
!rmdir "unzipped/dataset1/"

In [35]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.2 MB/s eta 0:00:00


In [56]:
# ==========================================
# COUNTERFEIT DETECTION V10 - VIEW-AWARE
# ==========================================

import os
import base64
import json
import glob
from groq import Groq, RateLimitError
from PIL import Image
import io
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
BASE_DIR = "unzipped"
OUTPUT_FILE = "counterfeit_validation_results_v6.jsonl"
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

FOLDER_MAP = {
    "all_helmet_images": "helmet",
    "all_airfilter_images": "air_filter",
    "all_sparkplug_images": "spark_plug"
}

# ==========================================
# API KEY MANAGEMENT (same as before)
# ==========================================
def get_api_keys():
    raw_keys = ""
    try:
        from google.colab import userdata
        raw_keys = userdata.get("GROQ_API_KEYS")
    except:
        try:
            from google.colab import userdata
            raw_keys = userdata.get("GROQ_API_KEY")
        except:
            print("❌ No secrets found.")
            return []

    keys = [k.strip() for k in raw_keys.split(',') if k.strip()]
    print(f"🔑 Loaded {len(keys)} API Key(s).")
    return keys

api_keys = get_api_keys()
clients = [Groq(api_key=k) for k in api_keys]
current_key_idx = 0

def get_next_client():
    global current_key_idx
    current_key_idx = (current_key_idx + 1) % len(clients)
    return clients[current_key_idx]

# ==========================================
# V10 PROMPTS (View-Aware)
# ==========================================

PROMPTS = {
    "helmet": {
        "system": """You are a motorcycle helmet quality inspector.
CRITICAL: "Cannot see feature" ≠ "Low quality"
Rate based ONLY on what IS visible. Ignore what isn't.
""",
        "user": """
## STEP 1: CLASSIFY THE VIEW TYPE

Determine the image type:
- EXTERIOR: Outside of helmet shell visible
- INTERIOR: Inside padding/liner visible
- CLOSE_UP: Zoomed on label/strap/vent/detail
- PACKAGING: Box or bag (helmet may not be visible)
- MIXED: Multiple views

## STEP 2: VIEW-SPECIFIC ASSESSMENT

### IF EXTERIOR:
- Assess: Shell finish, symmetry, visor quality, visible branding
- Ignore: Interior padding, internal labels

### IF INTERIOR:
- Assess: Padding thickness (>25mm=good), strap rivets (metal=good), liner quality
- Ignore: Shell exterior, visor
- Thick EPS foam + metal rivets = HIGH quality interior

### IF CLOSE_UP (Label):
- Assess: Text clarity, spelling, certification marks
- Clear ISI/DOT/ECE label = POSITIVE indicator → can be HIGH quality

### IF PACKAGING:
- Assess: Print quality, brand consistency, certification claims
- Professional packaging with certs = HIGH quality packaging

## STEP 3: RATING LOGIC

⚠️ IMPORTANT:
- If visible features look GOOD → "HIGH" (even if other features not visible)
- Only rate "LOW" if you see ACTUAL defects
- If truly nothing assessable → "UNCERTAIN"

DO NOT rate "LOW" just because view is limited!

## OUTPUT JSON:
{
  "view_type": "EXTERIOR | INTERIOR | CLOSE_UP | PACKAGING | MIXED",
  "helmet_style": "FULL_FACE | OPEN_FACE | HALF_SHELL | UNKNOWN",
  "assessed_features": ["list what you COULD assess"],
  "not_in_frame": ["list what you could NOT see - DO NOT PENALIZE THESE"],
  "indicators": {
    "primary_visible_quality": "HIGH | LOW | CANNOT_ASSESS",
    "branding": "PROFESSIONAL | SUSPICIOUS | NONE_VISIBLE",
    "certifications_found": ["DOT", "ECE", "ISI", "SNELL", "CCC"] or []
  },
  "visible_defects": ["only ACTUAL defects seen"],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "View type is [X]. Assessed [features]. Quality is [Y] because [Z]."
}
"""
    },

    "spark_plug": {
        "system": """You are a spark plug quality inspector.
RULE: "Not visible" ≠ "Defective". Rate ONLY what you can see.""",
        "user": """
## STEP 1: VIEW TYPE
- FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL

## STEP 2: ASSESS VISIBLE FEATURES
- Electrode tip: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Insulator: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Metal shell: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Branding: PROFESSIONAL / POOR / NOT_VISIBLE

## STEP 3: RATING
- Visible features good → "HIGH"
- Visible defects → "LOW"
- Can't assess enough → "UNCERTAIN"

{
  "view_type": "FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL",
  "brand_detected": "string or null",
  "assessed_features": ["what you could see"],
  "indicators": {
    "electrode": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "insulator": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "metal_shell": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "branding": "PROFESSIONAL | POOR | NOT_VISIBLE"
  },
  "visible_defects": ["actual defects only"],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "Based on [visible features], quality is [X]"
}
"""
    },

    "air_filter": {
        "system": """You are an air filter quality inspector.
RULE: Rate ONLY visible features. "Not visible" ≠ "Defective".""",
        "user": """
## VIEW TYPE: FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL

## ASSESS VISIBLE FEATURES:
- Pleats: HIGH_QUALITY (uniform) / LOW_QUALITY (wavy) / NOT_VISIBLE
- Frame: HIGH_QUALITY (clean) / LOW_QUALITY (flash) / NOT_VISIBLE
- Seal: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Branding: PROFESSIONAL / POOR / NOT_VISIBLE

## RATING: Based on VISIBLE features only

{
  "view_type": "FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL",
  "brand_detected": "string or null",
  "assessed_features": ["visible items"],
  "indicators": {
    "pleat_structure": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "frame_housing": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "seal_gasket": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "branding": "PROFESSIONAL | POOR | NOT_VISIBLE"
  },
  "visible_defects": [],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "explanation"
}
"""
    }
}

# ==========================================
# IMAGE ENCODING
# ==========================================
def encode_image(image_path, max_size_mb=3.5):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB': img = img.convert('RGB')
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            size_mb = buffer.tell() / (1024 * 1024)
            while size_mb > max_size_mb:
                width, height = img.size
                img = img.resize((int(width * 0.75), int(height * 0.75)), Image.Resampling.LANCZOS)
                buffer = io.BytesIO()
                img.save(buffer, format="JPEG", quality=85)
                size_mb = buffer.tell() / (1024 * 1024)
            return base64.b64encode(buffer.getvalue()).decode('utf-8')
    except Exception as e:
        print(f"⚠️ Image Error {image_path}: {e}")
        return None

# ==========================================
# MAIN EXECUTION
# ==========================================
tasks = []
for folder_name, prompt_key in FOLDER_MAP.items():
    full_path = os.path.join(BASE_DIR, folder_name)
    if os.path.exists(full_path):
        files = glob.glob(os.path.join(full_path, "*.*"))
        images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        for img in images:
            tasks.append({"path": img, "type": prompt_key})

print(f"🚀 Processing {len(tasks)} images with V10 prompts...")


🔑 Loaded 6 API Key(s).
🚀 Processing 274 images with V10 prompts...


In [57]:
processed_imgpaths=set()
with open('./counterfeit_validation_results_v5.jsonl', 'r', encoding='utf-8') as in_file:
  for line in in_file:
    data = json.loads(line)
    image_path = data["folder"] +"/"+ data['filename']
    processed_imgpaths.add(image_path)

In [58]:
with open(OUTPUT_FILE, "a") as f_out:
    for task in tqdm(tasks):
        img_path = task["path"]
        if img_path in processed_imgpaths: continue
        part_type = task["type"]
        filename = os.path.basename(img_path)
        prompt_data = PROMPTS[part_type]

        base64_image = encode_image(img_path)
        if not base64_image: continue

        success = False
        attempts = 0
        max_attempts = len(clients) * 2

        while not success and attempts < max_attempts:
            client = clients[current_key_idx]
            try:
                completion = client.chat.completions.create(
                    model=MODEL_ID,
                    messages=[
                        {"role": "system", "content": prompt_data["system"]},
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt_data["user"]},
                                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                            ]
                        }
                    ],
                    temperature=0.1,
                    response_format={"type": "json_object"},
                    stream=False
                )

                result = {
                    "filename": filename,
                    "folder": os.path.dirname(img_path),
                    "part_type": part_type,
                    "prompt_version": "v10",
                    "analysis": json.loads(completion.choices[0].message.content)
                }
                f_out.write(json.dumps(result) + "\n")
                f_out.flush()
                success = True

            except RateLimitError:
                client = get_next_client()
                attempts += 1
            except Exception as e:
                print(f"❌ Error on {filename}: {str(e)}")
                break

print(f"✅ Done. Results saved to {OUTPUT_FILE}")

  0%|          | 0/274 [00:00<?, ?it/s]

❌ Error on 3.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 57.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 80.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 42.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 45.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 

In [55]:
import os

base_directory = 'unzipped'
file_count = 0

for root, dirs, files in os.walk(base_directory):
    file_count += len(files)

print(f"Total number of files in '{base_directory}' and its subdirectories: {file_count}")

Total number of files in 'unzipped' and its subdirectories: 277


In [41]:
!pip install -q groq

import os
import base64
import json
import io
from PIL import Image
from groq import Groq

# --- CONFIGURATION ---
# 1. Paste ONE of your API keys here for the test
TEST_API_KEY ="***REMOVED_GROQ_KEY***"

# 2. Pick a test image path from your unzipped folder
# Example: "unzipped/all_helmet_images/helmet_123.jpg"
TEST_IMAGE_PATH = "unzipped/all_sparkplug_images/1.jpg"

# 3. Choose the part type for the test: "spark_plug", "helmet", or "air_filter"
TEST_PART_TYPE = "spark_plug"
# ---------------------

# Setup Client
client = Groq(api_key=TEST_API_KEY)
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

# Verified Prompt (Spark Plug Example - matches v5)
PROMPTS = {
    "spark_plug": {
        "system": "You are an expert Quality Control Inspector trained by Niterra (NGK/NTK). Your job is to identify counterfeit spark plugs using visual defects.",
        "user": """Analyze this image for the 5 OFFICIAL NGK Anti-Counterfeit checks. Output strictly in JSON:
        1. LOT CODE: Is a 4-digit alphanumeric code stamped on the hex? (Pass/Fail)
        2. CAPTIVE GASKET: Is the metal washer loose/threaded down (Fail) or captive (Pass)?
        3. MACHINING MARKS: Are there horizontal lathe lines on the C-Groove or Crimping? (Fail)
        4. ELECTRODE TIP: Is the tip fine-wire Iridium/Platinum (Pass) or thick/blunt Nickel (Fail)?
        5. BRANDING QUALITY: Is the logo crisp and centered (Pass) or smudged/off-center (Fail)?

        Output format: {"classification": "GENUINE|COUNTERFEIT|UNCERTAIN", "confidence": 0.0-1.0, "primary_reason": "string", "checks": {"lot_code": "pass/fail", "gasket": "pass/fail", "machining": "pass/fail", "electrode": "pass/fail", "branding": "pass/fail"}}"""
    }
    # (Other prompts omitted for brevity in test)
}

def encode_image(image_path):
    with Image.open(image_path) as img:
        if img.mode != 'RGB': img = img.convert('RGB')
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return base64.b64encode(buffer.getvalue()).decode('utf-8')

print(f"🧪 Testing on: {TEST_IMAGE_PATH}")

try:
    # Prepare
    base64_image = encode_image(TEST_IMAGE_PATH)
    prompt_data = PROMPTS[TEST_PART_TYPE]

    # Call API
    completion = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": prompt_data["system"]},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_data["user"]},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                ]
            }
        ],
        temperature=0.1,
        response_format={"type": "json_object"}
    )

    # Print Result
    print("\n✅ API RESPONSE:")
    print(json.dumps(json.loads(completion.choices[0].message.content), indent=2))

except Exception as e:
    print(f"\n❌ TEST FAILED: {e}")

🧪 Testing on: unzipped/all_sparkplug_images/1.jpg

✅ API RESPONSE:
{
  "classification": "COUNTERFEIT",
  "confidence": 1.0,
  "primary_reason": "The spark plug in the image is branded as BOSCH, not NGK, and does not match NGK's design.",
  "checks": {
    "lot_code": "fail",
    "gasket": "fail",
    "machining": "fail",
    "electrode": "fail",
    "branding": "fail"
  }
}


In [85]:
# ==========================================
# COUNTERFEIT DETECTION V10 - OPENROUTER
# ==========================================

import os
import base64
import json
import glob
import time
from openai import OpenAI
from PIL import Image
import io
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
BASE_DIR = "unzippp"
OUTPUT_FILE = "counterfeit_validation_results_v8_openrouter.jsonl"

# OpenRouter model options (pick one)
MODEL_ID = "nvidia/nemotron-nano-12b-v2-vl:free"  # Free tier
# MODEL_ID = "meta-llama/llama-4-maverick:free"  # Alternative free
# MODEL_ID = "google/gemini-2.0-flash-001"  # Paid but cheap
# MODEL_ID = "anthropic/claude-3.5-sonnet"  # Higher quality, paid

FOLDER_MAP = {
    "all_helmet_images": "helmet",
    "all_airfilter_images": "air_filter",
    "all_sparkplug_images": "spark_plug"
}

# ==========================================
# OPENROUTER API KEY MANAGEMENT
# ==========================================

def get_api_keys():
    """Retrieves OpenRouter keys from Colab/Kaggle secrets."""
    raw_keys = ""

    # Try Colab secrets first
    try:
        from google.colab import userdata
        raw_keys = userdata.get("OPENROUTER_APIKEYS")
        if not raw_keys:
            raw_keys = userdata.get("OPENROUTER_API_KEY")
    except:
        pass

    # Try environment variable
    if not raw_keys:
        raw_keys = os.environ.get("OPENROUTER_API_KEYS", "")
        if not raw_keys:
            raw_keys = os.environ.get("OPENROUTER_API_KEY", "")

    if not raw_keys:
        print("❌ No OpenRouter API keys found!")
        print("   Set OPENROUTER_API_KEYS in Colab secrets or environment")
        return []

    keys = [k.strip() for k in raw_keys.split(',') if k.strip()]
    print(f"🔑 Loaded {len(keys)} OpenRouter API Key(s)")
    return keys

def create_openrouter_client(api_key):
    """Create OpenRouter client using OpenAI SDK."""
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        default_headers={
            "HTTP-Referer": "https://github.com/counterfeit-detection",  # Optional
            "X-Title": "Counterfeit Parts Detection Pipeline"  # Optional
        }
    )

# Initialize clients
api_keys = get_api_keys()
clients = [create_openrouter_client(k) for k in api_keys]
current_key_idx = 0

def get_next_client():
    """Rotate to next available client."""
    global current_key_idx
    current_key_idx = (current_key_idx + 1) % len(clients)
    print(f"   🔄 Rotated to key {current_key_idx + 1}/{len(clients)}")
    return clients[current_key_idx]

# ==========================================
# V10 PROMPTS (View-Aware) - UNCHANGED
# ==========================================

PROMPTS = {
    "helmet": {
        "system": """You are a motorcycle helmet quality inspector.
CRITICAL: "Cannot see feature" ≠ "Low quality"
Rate based ONLY on what IS visible. Ignore what isn't.
""",
        "user": """
## STEP 1: CLASSIFY THE VIEW TYPE

Determine the image type:
- EXTERIOR: Outside of helmet shell visible
- INTERIOR: Inside padding/liner visible
- CLOSE_UP: Zoomed on label/strap/vent/detail
- PACKAGING: Box or bag (helmet may not be visible)
- MIXED: Multiple views

## STEP 2: VIEW-SPECIFIC ASSESSMENT

### IF EXTERIOR:
- Assess: Shell finish, symmetry, visor quality, visible branding
- Ignore: Interior padding, internal labels

### IF INTERIOR:
- Assess: Padding thickness (>25mm=good), strap rivets (metal=good), liner quality
- Ignore: Shell exterior, visor
- Thick EPS foam + metal rivets = HIGH quality interior

### IF CLOSE_UP (Label):
- Assess: Text clarity, spelling, certification marks
- Clear ISI/DOT/ECE label = POSITIVE indicator → can be HIGH quality

### IF PACKAGING:
- Assess: Print quality, brand consistency, certification claims
- Professional packaging with certs = HIGH quality packaging

## STEP 3: RATING LOGIC

⚠️ IMPORTANT:
- If visible features look GOOD → "HIGH" (even if other features not visible)
- Only rate "LOW" if you see ACTUAL defects
- If truly nothing assessable → "UNCERTAIN"

DO NOT rate "LOW" just because view is limited!

## OUTPUT JSON:
{
  "view_type": "EXTERIOR | INTERIOR | CLOSE_UP | PACKAGING | MIXED",
  "helmet_style": "FULL_FACE | OPEN_FACE | HALF_SHELL | UNKNOWN",
  "assessed_features": ["list what you COULD assess"],
  "not_in_frame": ["list what you could NOT see - DO NOT PENALIZE THESE"],
  "indicators": {
    "primary_visible_quality": "HIGH | LOW | CANNOT_ASSESS",
    "branding": "PROFESSIONAL | SUSPICIOUS | NONE_VISIBLE",
    "certifications_found": ["DOT", "ECE", "ISI", "SNELL", "CCC"] or []
  },
  "visible_defects": ["only ACTUAL defects seen"],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "View type is [X]. Assessed [features]. Quality is [Y] because [Z]."
}
"""
    },

    "spark_plug": {
        "system": """You are a spark plug quality inspector.
RULE: "Not visible" ≠ "Defective". Rate ONLY what you can see.""",
        "user": """
## STEP 1: VIEW TYPE
- FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL

## STEP 2: ASSESS VISIBLE FEATURES
- Electrode tip: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Insulator: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Metal shell: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Branding: PROFESSIONAL / POOR / NOT_VISIBLE

## STEP 3: RATING
- Visible features good → "HIGH"
- Visible defects → "LOW"
- Can't assess enough → "UNCERTAIN"

{
  "view_type": "FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL",
  "brand_detected": "string or null",
  "assessed_features": ["what you could see"],
  "indicators": {
    "electrode": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "insulator": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "metal_shell": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "branding": "PROFESSIONAL | POOR | NOT_VISIBLE"
  },
  "visible_defects": ["actual defects only"],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "Based on [visible features], quality is [X]"
}
"""
    },

    "air_filter": {
        "system": """You are an air filter quality inspector.
RULE: Rate ONLY visible features. "Not visible" ≠ "Defective".""",
        "user": """
## VIEW TYPE: FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL

## ASSESS VISIBLE FEATURES:
- Pleats: HIGH_QUALITY (uniform) / LOW_QUALITY (wavy) / NOT_VISIBLE
- Frame: HIGH_QUALITY (clean) / LOW_QUALITY (flash) / NOT_VISIBLE
- Seal: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Branding: PROFESSIONAL / POOR / NOT_VISIBLE

## RATING: Based on VISIBLE features only

{
  "view_type": "FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL",
  "brand_detected": "string or null",
  "assessed_features": ["visible items"],
  "indicators": {
    "pleat_structure": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "frame_housing": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "seal_gasket": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "branding": "PROFESSIONAL | POOR | NOT_VISIBLE"
  },
  "visible_defects": [],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "explanation"
}
"""
    }
}

# ==========================================
# IMAGE ENCODING
# ==========================================

def encode_image(image_path, max_size_mb=3.5):
    """Resize and base64 encode image for API."""
    try:
        with Image.open(image_path) as img:
            # Convert to RGB if needed
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # Initial encode
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            size_mb = buffer.tell() / (1024 * 1024)

            # Resize if too large
            while size_mb > max_size_mb:
                width, height = img.size
                img = img.resize(
                    (int(width * 0.75), int(height * 0.75)),
                    Image.Resampling.LANCZOS
                )
                buffer = io.BytesIO()
                img.save(buffer, format="JPEG", quality=85)
                size_mb = buffer.tell() / (1024 * 1024)

            return base64.b64encode(buffer.getvalue()).decode('utf-8')

    except Exception as e:
        print(f"⚠️ Image Error {image_path}: {e}")
        return None

# ==========================================
# OPENROUTER API CALL WITH RETRY
# ==========================================

def analyze_image(image_path, part_type, max_retries=None):
    """
    Analyze single image via OpenRouter API with key rotation.
    """
    global current_key_idx

    if max_retries is None:
        max_retries = len(clients) * 2

    prompt_data = PROMPTS[part_type]
    base64_image = encode_image(image_path)

    if not base64_image:
        return None

    attempts = 0
    last_error = None

    while attempts < max_retries:
        client = clients[current_key_idx]

        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {
                        "role": "system",
                        "content": prompt_data["system"]
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": prompt_data["user"]
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                }
                            }
                        ]
                    }
                ],
                temperature=0.1,
                max_tokens=1024,
                response_format={"type": "json_object"}
            )

            # Parse response
            response_text = completion.choices[0].message.content

            # Handle potential JSON parsing issues
            try:
                analysis = json.loads(response_text)
            except json.JSONDecodeError:
                # Try to extract JSON from response
                import re
                json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
                if json_match:
                    analysis = json.loads(json_match.group())
                else:
                    raise ValueError("Could not parse JSON from response")

            return {
                "filename": os.path.basename(image_path),
                "folder": os.path.dirname(image_path),
                "part_type": part_type,
                "prompt_version": "v10",
                "model": MODEL_ID,
                "analysis": analysis
            }

        except Exception as e:
            error_str = str(e).lower()
            last_error = e

            # Rate limit - rotate key and retry
            if any(x in error_str for x in ['429', 'rate limit', 'rate_limit', 'too many']):
                get_next_client()
                attempts += 1
                time.sleep(1)  # Brief pause
                continue

            # Quota exceeded - rotate key
            elif any(x in error_str for x in ['quota', 'exceeded', 'insufficient']):
                print(f"   ⚠️ Key {current_key_idx + 1} quota exceeded")
                get_next_client()
                attempts += 1
                continue

            # Server error - retry with backoff
            elif any(x in error_str for x in ['500', '502', '503', 'server error']):
                wait_time = 2 ** attempts
                print(f"   ⚠️ Server error, waiting {wait_time}s...")
                time.sleep(wait_time)
                attempts += 1
                continue

            # Other error - log and break
            else:
                print(f"❌ Error on {os.path.basename(image_path)}: {e}")
                break

    # All retries failed
    return {
        "filename": os.path.basename(image_path),
        "folder": os.path.dirname(image_path),
        "part_type": part_type,
        "error": str(last_error)
    }

# ==========================================
# LOAD EXISTING PROGRESS
# ==========================================

def load_processed_files(output_file):
    """Load already processed filenames to support resume."""
    processed = set()
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            for line in f:
                if line.strip():
                    try:
                        data = json.loads(line)
                        processed.add(data.get('filename', ''))
                    except:
                        continue
    return processed

# ==========================================
# MAIN EXECUTION
# ==========================================

def main():
    # Gather all image tasks
    tasks = []
    for folder_name, prompt_key in FOLDER_MAP.items():
        full_path = os.path.join(BASE_DIR, folder_name)
        if os.path.exists(full_path):
            files = glob.glob(os.path.join(full_path, "*.*"))
            images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
            for img in images:
                tasks.append({"path": img, "type": prompt_key})
            print(f"📁 {folder_name}: {len(images)} images")
        else:
            print(f"⚠️ Folder not found: {full_path}")

    print(f"\n🚀 Total: {len(tasks)} images to process")
    print(f"🤖 Model: {MODEL_ID}")
    print(f"🔑 API Keys: {len(clients)}")

    # Load already processed (for resume support)
    processed = load_processed_files(OUTPUT_FILE)
    if processed:
        print(f"⏩ Resuming: {len(processed)} already done")
        tasks = [t for t in tasks if os.path.basename(t['path']) not in processed]
        print(f"📋 Remaining: {len(tasks)} images")

    if not tasks:
        print("✅ All images already processed!")
        return

    # Process images
    success_count = 0
    error_count = 0

    with open(OUTPUT_FILE, "a") as f_out:
        for task in tqdm(tasks, desc="Processing"):
            result = analyze_image(task["path"], task["type"])

            if result:
                if "error" not in result:
                    success_count += 1
                else:
                    error_count += 1

                f_out.write(json.dumps(result) + "\n")
                f_out.flush()

            # Small delay to be nice to the API
            time.sleep(0.1)

    # Summary
    print("\n" + "=" * 50)
    print("📊 PROCESSING COMPLETE")
    print("=" * 50)
    print(f"✅ Successful: {success_count}")
    print(f"❌ Errors: {error_count}")
    print(f"💾 Results: {OUTPUT_FILE}")

# Run
if __name__ == "__main__":
    main()

🔑 Loaded 14 OpenRouter API Key(s)
📁 all_helmet_images: 158 images
📁 all_airfilter_images: 117 images
📁 all_sparkplug_images: 169 images

🚀 Total: 444 images to process
🤖 Model: nvidia/nemotron-nano-12b-v2-vl:free
🔑 API Keys: 14
⏩ Resuming: 444 already done
📋 Remaining: 0 images
✅ All images already processed!


In [96]:
unique_kv=set()
with open('./counterfeit_validation_results_v8_openrouter.jsonl', 'r', encoding='utf-8') as in_file:
  for line in in_file:
    data = json.loads(line)
    cur_list = []
    if data.get('analysis'):
      for k,v in data['analysis'].items():
        cur_key_type = k + ":" + str(type(v))
        cur_list.append(cur_key_type)
      if str(cur_list) not in  unique_kv: unique_kv.add(str(cur_list))
    else: pass
      # print(type(v))
      # print(k, ':',v, '\n----')
    # break
for item in unique_kv: print(item)

["view_type:<class 'str'>", "brand_detected:<class 'str'>", "assessed_features:<class 'list'>", "indicators:<class 'dict'>", "visible_defects:<class 'list'>", "visual_quality:<class 'str'>", "confidence:<class 'float'>", "reasoning:<class 'str'>"]
["view_type:<class 'str'>", "helmet_style:<class 'str'>", "assessed_features:<class 'list'>", "not_in_frame:<class 'list'>", "indicators:<class 'dict'>", "visual_quality:<class 'str'>", "confidence:<class 'float'>", "reasoning:<class 'str'>"]
["view_type:<class 'str'>", "brand_detected:<class 'NoneType'>", "assessed_features:<class 'list'>", "indicators:<class 'dict'>", "visible_defects:<class 'list'>", "visual_quality:<class 'str'>", "confidence:<class 'float'>", "reasoning:<class 'str'>"]
["view_type:<class 'str'>", "helmet_style:<class 'str'>", "assessed_features:<class 'list'>", "not_in_frame:<class 'list'>", "indicators:<class 'dict'>", "confidence:<class 'float'>", "reasoning:<class 'str'>"]
["view_type:<class 'str'>", "helmet_style:<cl

In [99]:
import json
from typing import Dict, Any, Optional

def normalize_analysis(data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Normalize VLM analysis output to consistent schema.
    Handles missing fields, empty analysis, and type inconsistencies.
    """

    analysis = data.get('analysis', {})
    part_type = data.get('part_type', 'unknown')

    # ==========================================
    # HANDLE EMPTY/MALFORMED ANALYSIS
    # ==========================================

    if not analysis or not isinstance(analysis, dict) or len(analysis) == 0:
        # Mark as failed - needs re-processing
        data['analysis'] = get_empty_analysis(part_type)
        data['processing_status'] = 'FAILED_EMPTY_RESPONSE'
        data['schema_version'] = 'normalized_v1'
        return data

    # Check if analysis has minimum required fields
    min_required = {'view_type', 'confidence', 'reasoning'}
    if not min_required.issubset(set(analysis.keys())):
        # Partial response - try to salvage what we can
        data['analysis'] = salvage_partial_analysis(analysis, part_type)
        data['processing_status'] = 'PARTIAL_RESPONSE'
        data['schema_version'] = 'normalized_v1'
        return data

    # ==========================================
    # NORMAL NORMALIZATION
    # ==========================================

    normalized = {
        # Common fields (all part types)
        "view_type": str(analysis.get('view_type', 'UNKNOWN')),
        "assessed_features": ensure_list(analysis.get('assessed_features', [])),
        "not_in_frame": ensure_list(analysis.get('not_in_frame', [])),
        "indicators": normalize_indicators(analysis.get('indicators', {}), part_type),
        "visible_defects": ensure_list(analysis.get('visible_defects', [])),
        "visual_quality": str(analysis.get('visual_quality', infer_quality(analysis))),
        "confidence": ensure_float(analysis.get('confidence', 0.5)),
        "reasoning": str(analysis.get('reasoning', '')),
    }

    # Part-specific fields
    if part_type == 'helmet':
        normalized["helmet_style"] = str(analysis.get('helmet_style', 'UNKNOWN'))
    else:
        normalized["brand_detected"] = analysis.get('brand_detected', None)

    data['analysis'] = normalized
    data['processing_status'] = 'SUCCESS'
    data['schema_version'] = 'normalized_v1'

    return data


def get_empty_analysis(part_type: str) -> Dict:
    """Return empty analysis template for failed records."""

    base = {
        "view_type": "UNKNOWN",
        "assessed_features": [],
        "not_in_frame": [],
        "indicators": get_default_indicators(part_type),
        "visible_defects": [],
        "visual_quality": "FAILED",
        "confidence": 0.0,
        "reasoning": "API returned empty response - needs re-processing"
    }

    if part_type == 'helmet':
        base["helmet_style"] = "UNKNOWN"
    else:
        base["brand_detected"] = None

    return base


def salvage_partial_analysis(analysis: Dict, part_type: str) -> Dict:
    """Try to salvage partial/malformed analysis."""

    salvaged = get_empty_analysis(part_type)
    salvaged["reasoning"] = "Partial API response - salvaged available fields"

    # Copy over any valid fields
    if 'view_type' in analysis and isinstance(analysis['view_type'], str):
        salvaged['view_type'] = analysis['view_type']

    if 'assessed_features' in analysis:
        salvaged['assessed_features'] = ensure_list(analysis['assessed_features'])

    if 'visible_defects' in analysis:
        salvaged['visible_defects'] = ensure_list(analysis['visible_defects'])

    if 'confidence' in analysis:
        salvaged['confidence'] = ensure_float(analysis['confidence'])

    if 'reasoning' in analysis and isinstance(analysis['reasoning'], str):
        salvaged['reasoning'] = analysis['reasoning']

    if 'indicators' in analysis and isinstance(analysis['indicators'], dict):
        salvaged['indicators'] = normalize_indicators(analysis['indicators'], part_type)

    # Try to infer quality
    salvaged['visual_quality'] = infer_quality(analysis) or 'UNCERTAIN'

    return salvaged


def ensure_list(value) -> list:
    """Ensure value is a list."""
    if isinstance(value, list):
        return value
    if value is None:
        return []
    if isinstance(value, str):
        return [value] if value else []
    return []


def ensure_float(value) -> float:
    """Ensure value is a float."""
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value)
        except:
            return 0.5
    return 0.5


def normalize_indicators(indicators: Dict, part_type: str) -> Dict:
    """Normalize indicators based on part type."""

    if not indicators or not isinstance(indicators, dict):
        return get_default_indicators(part_type)

    if part_type == 'helmet':
        return {
            "primary_visible_quality": str(indicators.get('primary_visible_quality', 'CANNOT_ASSESS')),
            "branding": str(indicators.get('branding', 'NONE_VISIBLE')),
            "certifications_found": ensure_list(indicators.get('certifications_found', []))
        }

    elif part_type == 'spark_plug':
        return {
            "electrode": str(indicators.get('electrode', 'NOT_VISIBLE')),
            "insulator": str(indicators.get('insulator', 'NOT_VISIBLE')),
            "metal_shell": str(indicators.get('metal_shell', 'NOT_VISIBLE')),
            "branding": str(indicators.get('branding', 'NOT_VISIBLE'))
        }

    elif part_type == 'air_filter':
        return {
            "pleat_structure": str(indicators.get('pleat_structure', 'NOT_VISIBLE')),
            "frame_housing": str(indicators.get('frame_housing', 'NOT_VISIBLE')),
            "seal_gasket": str(indicators.get('seal_gasket', 'NOT_VISIBLE')),
            "branding": str(indicators.get('branding', 'NOT_VISIBLE'))
        }

    return indicators


def get_default_indicators(part_type: str) -> Dict:
    """Get default indicators for part type."""

    defaults = {
        'helmet': {
            "primary_visible_quality": "CANNOT_ASSESS",
            "branding": "NONE_VISIBLE",
            "certifications_found": []
        },
        'spark_plug': {
            "electrode": "NOT_VISIBLE",
            "insulator": "NOT_VISIBLE",
            "metal_shell": "NOT_VISIBLE",
            "branding": "NOT_VISIBLE"
        },
        'air_filter': {
            "pleat_structure": "NOT_VISIBLE",
            "frame_housing": "NOT_VISIBLE",
            "seal_gasket": "NOT_VISIBLE",
            "branding": "NOT_VISIBLE"
        }
    }

    return defaults.get(part_type, {})


def infer_quality(analysis: Dict) -> str:
    """Infer visual_quality if missing from other fields."""

    if not analysis:
        return 'UNCERTAIN'

    if 'visual_quality' in analysis and analysis['visual_quality']:
        return str(analysis['visual_quality'])

    indicators = analysis.get('indicators', {})
    if not isinstance(indicators, dict):
        return 'UNCERTAIN'

    # For helmets
    primary_quality = str(indicators.get('primary_visible_quality', ''))
    if 'HIGH' in primary_quality.upper():
        return 'HIGH'
    elif 'LOW' in primary_quality.upper():
        return 'LOW'

    # For spark plugs / air filters
    quality_values = []
    for key in ['electrode', 'insulator', 'metal_shell', 'pleat_structure', 'frame_housing']:
        val = str(indicators.get(key, ''))
        if 'HIGH' in val.upper():
            quality_values.append('HIGH')
        elif 'LOW' in val.upper():
            quality_values.append('LOW')

    if quality_values:
        high_count = quality_values.count('HIGH')
        low_count = quality_values.count('LOW')
        if high_count > low_count:
            return 'HIGH'
        elif low_count > high_count:
            return 'LOW'

    # Check for visible defects
    defects = analysis.get('visible_defects', [])
    if isinstance(defects, list) and len(defects) > 0:
        return 'LOW'

    return 'UNCERTAIN'


# ==========================================
# BATCH PROCESSING WITH DETAILED STATS
# ==========================================

def normalize_jsonl_file(input_file: str, output_file: str):
    """Process entire JSONL file and normalize all schemas."""

    stats = {
        'total': 0,
        'success': 0,
        'failed_empty': 0,
        'partial': 0,
        'by_part_type': {},
        'failed_files': []
    }

    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:

        for line_num, line in enumerate(f_in, 1):
            stats['total'] += 1

            try:
                data = json.loads(line.strip())
                normalized = normalize_analysis(data)

                # Track status
                status = normalized.get('processing_status', 'UNKNOWN')
                if status == 'SUCCESS':
                    stats['success'] += 1
                elif status == 'FAILED_EMPTY_RESPONSE':
                    stats['failed_empty'] += 1
                    stats['failed_files'].append({
                        'line': line_num,
                        'filename': normalized.get('filename', 'unknown'),
                        'folder': normalized.get('folder', 'unknown')
                    })
                elif status == 'PARTIAL_RESPONSE':
                    stats['partial'] += 1

                # Track by part type
                pt = normalized.get('part_type', 'unknown')
                stats['by_part_type'][pt] = stats['by_part_type'].get(pt, 0) + 1

                f_out.write(json.dumps(normalized) + '\n')

            except json.JSONDecodeError as e:
                print(f"⚠️ JSON error on line {line_num}: {e}")
                stats['failed_empty'] += 1
            except Exception as e:
                print(f"⚠️ Error on line {line_num}: {e}")
                stats['failed_empty'] += 1

    # Print summary
    print("\n" + "=" * 60)
    print("📊 NORMALIZATION COMPLETE")
    print("=" * 60)
    print(f"✅ Total processed:     {stats['total']}")
    print(f"✅ Successful:          {stats['success']}")
    print(f"⚠️  Partial responses:   {stats['partial']}")
    print(f"❌ Failed/Empty:        {stats['failed_empty']}")

    print(f"\n📁 By part type:")
    for pt, count in stats['by_part_type'].items():
        print(f"   {pt}: {count}")

    if stats['failed_files']:
        print(f"\n❌ Failed files (need re-processing):")
        for item in stats['failed_files'][:20]:  # Show first 20
            print(f"   Line {item['line']}: {item['filename']}")

    return stats


# Run normalization
stats = normalize_jsonl_file(
    input_file='counterfeit_validation_results_v8_openrouter.jsonl',
    output_file='counterfeit_validation_results_v8_normalized.jsonl'
)


📊 NORMALIZATION COMPLETE
✅ Total processed:     444
✅ Successful:          432
⚠️  Partial responses:   0
❌ Failed/Empty:        12

📁 By part type:
   helmet: 158
   air_filter: 117
   spark_plug: 169

❌ Failed files (need re-processing):
   Line 65: hf_14.jpg
   Line 95: ho_73.jpg
   Line 121: hf_7.jpg
   Line 144: hf_48.jpg
   Line 192: _afo63.jpg
   Line 195: _afo65.jpg
   Line 222: _afo48.jpg
   Line 250: _aff20.jpg
   Line 266: _aff19.jpg
   Line 309: spf_46.jpg
   Line 369: spf_23.jpg
   Line 441: spf_53.jpg


In [100]:
def verify_normalized_schema(filepath: str):
    """Verify all records have consistent schema after normalization."""

    required_fields = {
        'view_type', 'assessed_features', 'not_in_frame',
        'indicators', 'visible_defects', 'visual_quality',
        'confidence', 'reasoning'
    }

    issues = []
    schema_check = set()
    status_counts = {'SUCCESS': 0, 'FAILED_EMPTY_RESPONSE': 0, 'PARTIAL_RESPONSE': 0}

    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            data = json.loads(line)
            analysis = data.get('analysis', {})
            status = data.get('processing_status', 'UNKNOWN')

            # Count by status
            status_counts[status] = status_counts.get(status, 0) + 1

            # Only check successful records for schema compliance
            if status == 'SUCCESS':
                missing = required_fields - set(analysis.keys())
                if missing:
                    issues.append(f"Line {i}: Missing {missing}")

                if not isinstance(analysis.get('assessed_features'), list):
                    issues.append(f"Line {i}: assessed_features not list")
                if not isinstance(analysis.get('visible_defects'), list):
                    issues.append(f"Line {i}: visible_defects not list")
                if not isinstance(analysis.get('confidence'), (int, float)):
                    issues.append(f"Line {i}: confidence not numeric")

            # Track unique schemas
            schema_sig = str(sorted(analysis.keys()))
            schema_check.add(schema_sig)

    print("=" * 60)
    print("📋 VERIFICATION REPORT")
    print("=" * 60)

    print(f"\n📊 Processing Status:")
    for status, count in status_counts.items():
        emoji = "✅" if status == "SUCCESS" else "⚠️" if status == "PARTIAL_RESPONSE" else "❌"
        print(f"   {emoji} {status}: {count}")

    print(f"\n📋 Unique schemas: {len(schema_check)}")
    for schema in schema_check:
        print(f"   {schema[:80]}...")

    if issues:
        print(f"\n⚠️ Schema issues in SUCCESS records: {len(issues)}")
        for issue in issues[:10]:
            print(f"   {issue}")
        return False
    else:
        print("\n✅ All SUCCESS records have valid schema!")
        return True


# Verify
verify_normalized_schema('counterfeit_validation_results_v8_normalized.jsonl')

📋 VERIFICATION REPORT

📊 Processing Status:
   ✅ SUCCESS: 432
   ❌ FAILED_EMPTY_RESPONSE: 12
   ⚠️ PARTIAL_RESPONSE: 0

📋 Unique schemas: 2
   ['assessed_features', 'confidence', 'helmet_style', 'indicators', 'not_in_frame'...
   ['assessed_features', 'brand_detected', 'confidence', 'indicators', 'not_in_fram...

✅ All SUCCESS records have valid schema!


True

In [101]:
# Option B: Exclude failed records from training
def create_clean_dataset(normalized_file, output_file):
    """Create dataset with only successful records."""

    clean_count = 0

    with open(normalized_file, 'r') as f_in, \
         open(output_file, 'w') as f_out:

        for line in f_in:
            data = json.loads(line)
            if data.get('processing_status') == 'SUCCESS':
                f_out.write(line)
                clean_count += 1

    print(f"✅ Created clean dataset: {clean_count} records")
    return clean_count

create_clean_dataset(
    'counterfeit_validation_results_v8_normalized.jsonl',
    'counterfeit_validation_CLEAN.jsonl'
)

✅ Created clean dataset: 432 records


432

In [ ]:
os.listdir('./unzippp/dataset1/air filter fake/')

In [87]:
def format_as_natural_response(result):
    """Convert to natural language assistant response."""

    analysis = result['analysis']
    part_type = result['part_type'].replace('_', ' ')

    # Build natural response
    quality = analysis['visual_quality']
    confidence = analysis['confidence']
    view = analysis['view_type'].lower()

    # Quality descriptor
    quality_desc = {
        "HIGH": "good manufacturing quality",
        "LOW": "concerning quality issues",
        "UNCERTAIN": "insufficient information for assessment"
    }.get(quality, "unknown quality")

    # Build response
    response = f"Based on my analysis of this {part_type} image, I can see this is an **{view} view**.\n\n"

    # What was assessed
    if analysis['assessed_features']:
        features = ', '.join(analysis['assessed_features'])
        response += f"**What I could assess:** {features}\n\n"

    # What wasn't visible
    if analysis['not_in_frame']:
        not_visible = ', '.join(analysis['not_in_frame'])
        response += f"**Not visible in this image:** {not_visible} (not penalized)\n\n"

    # Defects
    if analysis['visible_defects']:
        defects = ', '.join(analysis['visible_defects'])
        response += f"⚠️ **Defects detected:** {defects}\n\n"

    # Certifications
    certs = analysis['indicators'].get('certifications_found', [])
    if certs:
        response += f"✅ **Certifications found:** {', '.join(certs)}\n\n"

    # Final verdict
    emoji = {"HIGH": "✅", "LOW": "❌", "UNCERTAIN": "❓"}.get(quality, "❓")
    response += f"**Overall Quality:** {emoji} **{quality}** (Confidence: {confidence:.0%})\n\n"
    response += f"**Reasoning:** {analysis['reasoning']}"

    return {
        "messages": [
            {"role": "user", "content": f"Can you analyze this {part_type} image for quality?"},
            {"role": "assistant", "content": response}
        ]
    }

In [88]:
# Your result
result = {
    "filename": "hf_50.jpg",
    "folder": "unzippp/all_helmet_images",
    "part_type": "helmet",
    "prompt_version": "v10",
    "model": "nvidia/nemotron-nano-12b-v2-vl:free",
    "analysis": {
        "view_type": "INTERIOR",
        "helmet_style": "UNKNOWN",
        "assessed_features": ["padding stitching quality", "visible red component (possibly strap system)"],
        "not_in_frame": ["exterior shell", "full strap system", "certification labels", "full liner thickness"],
        "indicators": {
            "primary_visible_quality": "HIGH",
            "branding": "NONE_VISIBLE",
            "certifications_found": []
        },
        "visible_defects": [],
        "visual_quality": "HIGH",
        "confidence": 0.85,
        "reasoning": "View type is INTERIOR. Assessed padding stitching (neat, professional) and visible red component (likely part of strap system). No defects observed. Quality is HIGH because visible interior features show professional construction with no apparent flaws, though full assessment is limited by view constraints."
    }
}

# Convert to natural format
formatted = format_as_natural_response(result)
print(formatted['messages'][1]['content'])

Based on my analysis of this helmet image, I can see this is an **interior view**.

**What I could assess:** padding stitching quality, visible red component (possibly strap system)

**Not visible in this image:** exterior shell, full strap system, certification labels, full liner thickness (not penalized)

**Overall Quality:** ✅ **HIGH** (Confidence: 85%)

**Reasoning:** View type is INTERIOR. Assessed padding stitching (neat, professional) and visible red component (likely part of strap system). No defects observed. Quality is HIGH because visible interior features show professional construction with no apparent flaws, though full assessment is limited by view constraints.


In [71]:
import os

directory_path = './unzippp/dataset1/spark plug og'

# Get a list of all files in the directory
files = os.listdir(directory_path)

for filename in files:
    # Construct the old and new file paths
    old_file_path = os.path.join(directory_path, filename)
    new_filename = 'spo_' + filename
    new_file_path = os.path.join(directory_path, new_filename)

    # Rename the file
    os.rename(old_file_path, new_file_path)
    print(f"Renamed '{filename}' to '{new_filename}'")

print("All files renamed successfully!")

Renamed '3.jpg' to 'spo_3.jpg'
Renamed '57.jpg' to 'spo_57.jpg'
Renamed '97.jpg' to 'spo_97.jpg'
Renamed '80.jpg' to 'spo_80.jpg'
Renamed '61TF4nwfzzL._SX522_.jpg' to 'spo_61TF4nwfzzL._SX522_.jpg'
Renamed '103.jpg' to 'spo_103.jpg'
Renamed '42.jpg' to 'spo_42.jpg'
Renamed '45.jpg' to 'spo_45.jpg'
Renamed '101.jpg' to 'spo_101.jpg'
Renamed '38.jpg' to 'spo_38.jpg'
Renamed '66.jpg' to 'spo_66.jpg'
Renamed '52.jpg' to 'spo_52.jpg'
Renamed '69.jpg' to 'spo_69.jpg'
Renamed '30.jpg' to 'spo_30.jpg'
Renamed '35.jpg' to 'spo_35.jpg'
Renamed '86.jpg' to 'spo_86.jpg'
Renamed '74.jpg' to 'spo_74.jpg'
Renamed '83.jpg' to 'spo_83.jpg'
Renamed '14.jpg' to 'spo_14.jpg'
Renamed '8.jpg' to 'spo_8.jpg'
Renamed '68.jpg' to 'spo_68.jpg'
Renamed '73' to 'spo_73'
Renamed '11.jpg' to 'spo_11.jpg'
Renamed '61.jpg' to 'spo_61.jpg'
Renamed '75.jpg' to 'spo_75.jpg'
Renamed '81.jpg' to 'spo_81.jpg'
Renamed '53.jpg' to 'spo_53.jpg'
Renamed '76.jpg' to 'spo_76.jpg'
Renamed '28.jpg' to 'spo_28.jpg'
Renamed '27.jpg' 

In [78]:
!mkdir ./unzippp/all_sparkplug_images

In [81]:
!mv "./unzippp/dataset1/spark plug og"/*  ./unzippp/all_sparkplug_images

In [82]:
!rm -rf ./unzippp/dataset1/

In [104]:
import json
import os
from PIL import Image
from typing import Dict, List, Any
from tqdm import tqdm

# ==========================================
# CONFIGURATION
# ==========================================

INPUT_FILE = "counterfeit_validation_results_v8_normalized.jsonl"
OUTPUT_FILE = "counterfeit_finetuning_dataset.json"
IMAGE_BASE_DIR = ""  # Set if images are in different location

# Target image size (Unsloth recommends 300-1000px)
TARGET_SIZE = (512, 512)
RESIZE_IMAGES = True

# ==========================================
# INSTRUCTION TEMPLATES
# ==========================================

SYSTEM_INSTRUCTIONS = {
    "helmet": "You are an expert motorcycle helmet quality inspector. Analyze the image and assess manufacturing quality, safety indicators, and any visible defects.",

    "spark_plug": "You are an expert spark plug quality inspector. Analyze the image and assess manufacturing quality, electrode condition, and any visible defects.",

    "air_filter": "You are an expert air filter quality inspector. Analyze the image and assess manufacturing quality, pleat structure, and any visible defects."
}

USER_INSTRUCTIONS = {
    "helmet": "Analyze this motorcycle helmet image for quality assessment. Identify the view type, assess visible features, check for certifications, and provide a quality rating.",

    "spark_plug": "Analyze this spark plug image for quality assessment. Identify the view type, assess visible features like electrode and insulator condition, and provide a quality rating.",

    "air_filter": "Analyze this air filter image for quality assessment. Identify the view type, assess visible features like pleat structure and frame quality, and provide a quality rating."
}

# ==========================================
# RESPONSE FORMATTERS
# ==========================================

def format_helmet_response(analysis: Dict) -> str:
    """Format helmet analysis as natural language response."""

    response = f"""**Quality Assessment Report**

**View Type:** {analysis.get('view_type', 'UNKNOWN')}
**Helmet Style:** {analysis.get('helmet_style', 'UNKNOWN')}

**Assessed Features:**
{chr(10).join(f"• {feat}" for feat in analysis.get('assessed_features', []))}

**Not In Frame (Not Penalized):**
{chr(10).join(f"• {feat}" for feat in analysis.get('not_in_frame', []))}

**Quality Indicators:**
• Primary Quality: {analysis.get('indicators', {}).get('primary_visible_quality', 'N/A')}
• Branding: {analysis.get('indicators', {}).get('branding', 'N/A')}
• Certifications Found: {', '.join(analysis.get('indicators', {}).get('certifications_found', [])) or 'None visible'}

**Visible Defects:** {', '.join(analysis.get('visible_defects', [])) or 'None detected'}

**Overall Assessment:**
• **Quality Rating:** {analysis.get('visual_quality', 'UNCERTAIN')}
• **Confidence:** {analysis.get('confidence', 0.5):.0%}

**Reasoning:** {analysis.get('reasoning', 'No reasoning provided.')}"""

    return response


def format_spark_plug_response(analysis: Dict) -> str:
    """Format spark plug analysis as natural language response."""

    indicators = analysis.get('indicators', {})

    response = f"""**Quality Assessment Report**

**View Type:** {analysis.get('view_type', 'UNKNOWN')}
**Brand Detected:** {analysis.get('brand_detected') or 'Not visible'}

**Assessed Features:**
{chr(10).join(f"• {feat}" for feat in analysis.get('assessed_features', []))}

**Component Quality:**
• Electrode: {indicators.get('electrode', 'NOT_VISIBLE')}
• Insulator: {indicators.get('insulator', 'NOT_VISIBLE')}
• Metal Shell: {indicators.get('metal_shell', 'NOT_VISIBLE')}
• Branding: {indicators.get('branding', 'NOT_VISIBLE')}

**Visible Defects:** {', '.join(analysis.get('visible_defects', [])) or 'None detected'}

**Overall Assessment:**
• **Quality Rating:** {analysis.get('visual_quality', 'UNCERTAIN')}
• **Confidence:** {analysis.get('confidence', 0.5):.0%}

**Reasoning:** {analysis.get('reasoning', 'No reasoning provided.')}"""

    return response


def format_air_filter_response(analysis: Dict) -> str:
    """Format air filter analysis as natural language response."""

    indicators = analysis.get('indicators', {})

    response = f"""**Quality Assessment Report**

**View Type:** {analysis.get('view_type', 'UNKNOWN')}
**Brand Detected:** {analysis.get('brand_detected') or 'Not visible'}

**Assessed Features:**
{chr(10).join(f"• {feat}" for feat in analysis.get('assessed_features', []))}

**Component Quality:**
• Pleat Structure: {indicators.get('pleat_structure', 'NOT_VISIBLE')}
• Frame/Housing: {indicators.get('frame_housing', 'NOT_VISIBLE')}
• Seal/Gasket: {indicators.get('seal_gasket', 'NOT_VISIBLE')}
• Branding: {indicators.get('branding', 'NOT_VISIBLE')}

**Visible Defects:** {', '.join(analysis.get('visible_defects', [])) or 'None detected'}

**Overall Assessment:**
• **Quality Rating:** {analysis.get('visual_quality', 'UNCERTAIN')}
• **Confidence:** {analysis.get('confidence', 0.5):.0%}

**Reasoning:** {analysis.get('reasoning', 'No reasoning provided.')}"""

    return response


def format_json_response(analysis: Dict) -> str:
    """Format analysis as clean JSON response."""
    return json.dumps(analysis, indent=2)


# ==========================================
# IMAGE PROCESSING
# ==========================================

def load_and_resize_image(image_path: str, target_size: tuple = None) -> Image.Image:
    """Load image and optionally resize."""

    try:
        img = Image.open(image_path)

        # Convert to RGB if needed
        if img.mode != 'RGB':
            img = img.convert('RGB')

        # Resize if specified
        if target_size and RESIZE_IMAGES:
            img.thumbnail(target_size, Image.Resampling.LANCZOS)

        return img

    except Exception as e:
        print(f"⚠️ Error loading image {image_path}: {e}")
        return None


def get_image_path(data: Dict) -> str:
    """Get full image path from data record."""

    folder = data.get('folder', '')
    filename = data.get('filename', '')

    if IMAGE_BASE_DIR:
        return os.path.join(IMAGE_BASE_DIR, folder, filename)
    else:
        return os.path.join(folder, filename)


# ==========================================
# DATASET CONVERSION
# ==========================================

def convert_to_conversation(data: Dict, response_format: str = "natural") -> Dict:
    """
    Convert a single record to Unsloth vision fine-tuning format.

    Args:
        data: Normalized analysis record
        response_format: "natural" for readable text, "json" for structured

    Returns:
        Conversation in Unsloth format
    """

    # Skip failed records
    if data.get('processing_status') != 'SUCCESS':
        return None

    analysis = data.get('analysis', {})
    part_type = data.get('part_type', 'unknown')
    image_path = get_image_path(data)

    # Load image
    image = load_and_resize_image(image_path, TARGET_SIZE)
    if image is None:
        return None

    # Get instruction
    user_instruction = USER_INSTRUCTIONS.get(part_type,
        "Analyze this product image for manufacturing quality.")

    # Format response based on part type
    if response_format == "natural":
        if part_type == 'helmet':
            assistant_response = format_helmet_response(analysis)
        elif part_type == 'spark_plug':
            assistant_response = format_spark_plug_response(analysis)
        elif part_type == 'air_filter':
            assistant_response = format_air_filter_response(analysis)
        else:
            assistant_response = format_json_response(analysis)
    else:
        assistant_response = format_json_response(analysis)

    # Create conversation in Unsloth format
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_instruction},
                {"type": "image", "image": image}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": assistant_response}
            ]
        }
    ]

    return {
        "messages": conversation,
        "metadata": {
            "filename": data.get('filename'),
            "part_type": part_type,
            "view_type": analysis.get('view_type'),
            "quality": analysis.get('visual_quality'),
            "confidence": analysis.get('confidence')
        }
    }


def convert_to_conversation_with_system(data: Dict, response_format: str = "natural") -> Dict:
    """
    Convert with system message included.
    """

    if data.get('processing_status') != 'SUCCESS':
        return None

    analysis = data.get('analysis', {})
    part_type = data.get('part_type', 'unknown')
    image_path = get_image_path(data)

    image = load_and_resize_image(image_path, TARGET_SIZE)
    if image is None:
        return None

    system_instruction = SYSTEM_INSTRUCTIONS.get(part_type,
        "You are an expert quality inspector. Analyze product images for manufacturing quality.")
    user_instruction = USER_INSTRUCTIONS.get(part_type,
        "Analyze this product image for quality assessment.")

    # Format response
    if response_format == "natural":
        if part_type == 'helmet':
            assistant_response = format_helmet_response(analysis)
        elif part_type == 'spark_plug':
            assistant_response = format_spark_plug_response(analysis)
        elif part_type == 'air_filter':
            assistant_response = format_air_filter_response(analysis)
        else:
            assistant_response = format_json_response(analysis)
    else:
        assistant_response = format_json_response(analysis)

    conversation = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": system_instruction}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_instruction},
                {"type": "image", "image": image}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": assistant_response}
            ]
        }
    ]

    return {
        "messages": conversation,
        "metadata": {
            "filename": data.get('filename'),
            "part_type": part_type,
            "view_type": analysis.get('view_type'),
            "quality": analysis.get('visual_quality'),
            "confidence": analysis.get('confidence')
        }
    }


# ==========================================
# MAIN CONVERSION FUNCTION
# ==========================================

def create_finetuning_dataset(
    input_file: str,
    output_file: str = None,
    response_format: str = "natural",
    include_system: bool = False,
    min_confidence: float = 0.0
) -> List[Dict]:
    """
    Create Unsloth-compatible fine-tuning dataset.

    Args:
        input_file: Path to normalized JSONL file
        output_file: Optional path to save JSON output
        response_format: "natural" or "json"
        include_system: Whether to include system message
        min_confidence: Minimum confidence threshold

    Returns:
        List of converted conversations
    """

    dataset = []
    stats = {
        'total': 0,
        'converted': 0,
        'skipped_failed': 0,
        'skipped_low_conf': 0,
        'skipped_no_image': 0,
        'by_quality': {},
        'by_part': {},
        'by_view': {}
    }

    # Choose converter function
    converter = convert_to_conversation_with_system if include_system else convert_to_conversation

    # Load and convert
    print(f"📂 Loading from: {input_file}")

    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in tqdm(lines, desc="Converting"):
        stats['total'] += 1

        try:
            data = json.loads(line.strip())

            # Skip failed records
            if data.get('processing_status') != 'SUCCESS':
                stats['skipped_failed'] += 1
                continue

            # Skip low confidence
            conf = data.get('analysis', {}).get('confidence', 0)
            if conf < min_confidence:
                stats['skipped_low_conf'] += 1
                continue

            # Convert
            converted = converter(data, response_format)

            if converted is None:
                stats['skipped_no_image'] += 1
                continue

            dataset.append(converted)
            stats['converted'] += 1

            # Track stats
            meta = converted.get('metadata', {})
            q = meta.get('quality', 'UNKNOWN')
            p = meta.get('part_type', 'unknown')
            v = meta.get('view_type', 'UNKNOWN')

            stats['by_quality'][q] = stats['by_quality'].get(q, 0) + 1
            stats['by_part'][p] = stats['by_part'].get(p, 0) + 1
            stats['by_view'][v] = stats['by_view'].get(v, 0) + 1

        except Exception as e:
            print(f"⚠️ Error: {e}")
            continue

    # Print summary
    print("\n" + "=" * 60)
    print("📊 DATASET CREATION COMPLETE")
    print("=" * 60)
    print(f"Total records:        {stats['total']}")
    print(f"✅ Converted:          {stats['converted']}")
    print(f"❌ Skipped (failed):   {stats['skipped_failed']}")
    print(f"❌ Skipped (low conf): {stats['skipped_low_conf']}")
    print(f"❌ Skipped (no image): {stats['skipped_no_image']}")

    print(f"\n🏷️ By Quality:")
    for q, count in sorted(stats['by_quality'].items()):
        pct = count / stats['converted'] * 100 if stats['converted'] > 0 else 0
        print(f"   {q:12} {count:4} ({pct:5.1f}%)")

    print(f"\n🔧 By Part Type:")
    for p, count in sorted(stats['by_part'].items()):
        print(f"   {p:15} {count:4}")

    print(f"\n📸 By View Type:")
    for v, count in sorted(stats['by_view'].items()):
        print(f"   {v:15} {count:4}")

    # Note about saving
    if output_file:
        print(f"\n⚠️ Note: Cannot save PIL images to JSON directly.")
        print(f"   Dataset returned as list for direct use with Unsloth.")
        print(f"   Use the dataset directly in your training script.")

    return dataset


# ==========================================
# ALTERNATIVE: SAVE PATHS INSTEAD OF IMAGES
# ==========================================

def create_finetuning_dataset_with_paths(
    input_file: str,
    output_file: str,
    response_format: str = "natural",
    include_system: bool = False,
    min_confidence: float = 0.0
) -> List[Dict]:
    """
    Create dataset with image paths (for saving to JSON).
    Images will be loaded at training time.
    """

    dataset = []
    stats = {'total': 0, 'converted': 0, 'skipped': 0}

    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in tqdm(lines, desc="Converting"):
        stats['total'] += 1

        try:
            data = json.loads(line.strip())

            if data.get('processing_status') != 'SUCCESS':
                stats['skipped'] += 1
                continue

            conf = data.get('analysis', {}).get('confidence', 0)
            if conf < min_confidence:
                stats['skipped'] += 1
                continue

            analysis = data.get('analysis', {})
            part_type = data.get('part_type', 'unknown')
            image_path = get_image_path(data)

            # Check image exists
            if not os.path.exists(image_path):
                stats['skipped'] += 1
                continue

            # Get instruction
            system_instruction = SYSTEM_INSTRUCTIONS.get(part_type, "")
            user_instruction = USER_INSTRUCTIONS.get(part_type,
                "Analyze this product image for quality assessment.")

            # Format response
            if response_format == "natural":
                if part_type == 'helmet':
                    assistant_response = format_helmet_response(analysis)
                elif part_type == 'spark_plug':
                    assistant_response = format_spark_plug_response(analysis)
                elif part_type == 'air_filter':
                    assistant_response = format_air_filter_response(analysis)
                else:
                    assistant_response = format_json_response(analysis)
            else:
                assistant_response = format_json_response(analysis)

            # Create entry with image path
            if include_system:
                messages = [
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": user_instruction},
                    {"role": "assistant", "content": assistant_response}
                ]
            else:
                messages = [
                    {"role": "user", "content": user_instruction},
                    {"role": "assistant", "content": assistant_response}
                ]

            entry = {
                "messages": messages,
                "image": image_path,
                "metadata": {
                    "filename": data.get('filename'),
                    "part_type": part_type,
                    "view_type": analysis.get('view_type'),
                    "quality": analysis.get('visual_quality'),
                    "confidence": analysis.get('confidence')
                }
            }

            dataset.append(entry)
            stats['converted'] += 1

        except Exception as e:
            print(f"⚠️ Error: {e}")
            stats['skipped'] += 1

    # Save to JSON
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, indent=2)

    print(f"\n✅ Created {stats['converted']} training samples")
    print(f"💾 Saved to: {output_file}")

    return dataset


# ==========================================
# RUN CONVERSION
# ==========================================

if __name__ == "__main__":

    # Option 1: Create dataset with paths (saveable to JSON)
    # dataset_with_paths = create_finetuning_dataset_with_paths(
    #     input_file='/content/counterfeit_validation_CLEAN.jsonl',
    #     output_file='counterfeit_finetuning_dataset.json',
    #     response_format='natural',
    #     include_system=False,
    #     min_confidence=0.5
    # )

    # Option 2: Create dataset with PIL images (for direct training)
    dataset_with_images = create_finetuning_dataset(
        input_file='/content/counterfeit_validation_CLEAN.jsonl',
        response_format='natural',
        include_system=False,
        min_confidence=0.5
    )

📂 Loading from: /content/counterfeit_validation_CLEAN.jsonl


Converting: 100%|██████████| 432/432 [00:16<00:00, 26.49it/s]


📊 DATASET CREATION COMPLETE
Total records:        432
✅ Converted:          414
❌ Skipped (failed):   0
❌ Skipped (low conf): 18
❌ Skipped (no image): 0

🏷️ By Quality:
   HIGH          384 ( 92.8%)
   LOW            28 (  6.8%)
   LOW | HIGH      1 (  0.2%)
   UNCERTAIN       1 (  0.2%)

🔧 By Part Type:
   air_filter       111
   helmet           152
   spark_plug       151

📸 By View Type:
   CLOSE_UP         128
   CLOSE_UP | PACKAGING    1
   EXTERIOR         103
   FULL_PRODUCT     114
   FULL_PRODUCT | CLOSE_UP    3
   FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL    1
   FULL_PRODUCT | PACKAGING    7
   INTERIOR          34
   MIXED              7
   PACKAGING          4
   PARTIAL           12


In [103]:
def verify_unsloth_format(json_file: str) -> bool:
    """Verify dataset is ready for Unsloth training."""

    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    issues = []

    for i, item in enumerate(data):
        # Check required structure
        if 'messages' not in item:
            issues.append(f"Sample {i}: Missing 'messages'")
            continue

        if 'image' not in item:
            issues.append(f"Sample {i}: Missing 'image'")

        msgs = item['messages']

        # Check message count
        if len(msgs) < 2:
            issues.append(f"Sample {i}: Need user + assistant messages")
            continue

        # Check roles
        if msgs[0].get('role') != 'user':
            issues.append(f"Sample {i}: First message should be 'user'")
        if msgs[1].get('role') != 'assistant':
            issues.append(f"Sample {i}: Second message should be 'assistant'")

        # Check content exists
        if not msgs[0].get('content'):
            issues.append(f"Sample {i}: Empty user content")
        if not msgs[1].get('content'):
            issues.append(f"Sample {i}: Empty assistant content")

        # Check image path exists
        img_path = item.get('image', '')
        if img_path and not os.path.exists(img_path):
            issues.append(f"Sample {i}: Image not found: {img_path}")

    print("=" * 60)
    print("✅ UNSLOTH FORMAT VERIFICATION")
    print("=" * 60)

    if issues:
        print(f"\n⚠️ Found {len(issues)} issues:")
        for issue in issues[:20]:
            print(f"   {issue}")
        return False
    else:
        print(f"\n✅ All {len(data)} samples are valid!")
        print(f"✅ Dataset is ready for Unsloth fine-tuning!")
        return True

import os
verify_unsloth_format("counterfeit_finetuning_dataset.json")

✅ UNSLOTH FORMAT VERIFICATION

✅ All 414 samples are valid!
✅ Dataset is ready for Unsloth fine-tuning!


True

In [105]:
from datasets import Dataset, Image as HFImage
from huggingface_hub import login
import json
from google.colab import userdata

login(token=userdata.get('WRITE_HF_TOKEN'))

def prepare_full_unsloth_format(dataset_with_images: list) -> dict:
    """
    Store the exact format Unsloth expects, with image separate.
    When loading, reconstruct the full conversation.
    """

    hf_data = {
        "image": [],
        "user_text": [],
        "assistant_text": [],
        "filename": [],
        "part_type": [],
        "view_type": [],
        "quality": [],
        "confidence": []
    }

    for item in dataset_with_images:
        messages = item.get("messages", [])
        metadata = item.get("metadata", {})

        # Extract components
        user_content = messages[0]["content"] if messages else []
        image = None
        user_text = ""

        for content in user_content:
            if content.get("type") == "image":
                image = content.get("image")
            elif content.get("type") == "text":
                user_text = content.get("text", "")

        if image is None:
            continue

        asst_content = messages[1]["content"] if len(messages) > 1 else []
        asst_text = ""
        for content in asst_content:
            if content.get("type") == "text":
                asst_text = content.get("text", "")

        hf_data["image"].append(image)
        hf_data["user_text"].append(user_text)
        hf_data["assistant_text"].append(asst_text)
        hf_data["filename"].append(metadata.get("filename", ""))
        hf_data["part_type"].append(metadata.get("part_type", ""))
        hf_data["view_type"].append(metadata.get("view_type", ""))
        hf_data["quality"].append(metadata.get("quality", ""))
        hf_data["confidence"].append(float(metadata.get("confidence", 0.0)))

    return hf_data

# Prepare and push
hf_data = prepare_full_unsloth_format(dataset_with_images)
hf_dataset = Dataset.from_dict(hf_data).cast_column("image", HFImage())

hf_dataset.push_to_hub("barryallen16/bike_parts_counterfeit_detection")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/414 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   1%|1         |  526kB / 46.9MB            

CommitInfo(commit_url='https://huggingface.co/datasets/barryallen16/bike_parts_counterfeit_detection/commit/8ee9d3ee65c43713322476df6f5a7df4a9e54061', commit_message='Upload dataset', commit_description='', oid='8ee9d3ee65c43713322476df6f5a7df4a9e54061', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/barryallen16/bike_parts_counterfeit_detection', endpoint='https://huggingface.co', repo_type='dataset', repo_id='barryallen16/bike_parts_counterfeit_detection'), pr_revision=None, pr_num=None)